In [ ]:
import json
import glob
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mdtraj
from scipy.stats import spearmanr, pearsonr, rankdata

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "svg.fonttype": "none",
})

RES = "../../data/md_1us"  # 1-us MD trajectories, one frame per ns
DATA = "../../data/analysis/Fig4/per_protein_analysis"   # read by enrichment_structures/generate_pml_*
FIG = ("Figs")

# Same source/convention as compute_md_justs2_justdihedral_persist90_panav_10seedens.ipynb:
# PANAV/10-seed-ensemble acerr, no persist90 filter -- exposure/ss_class already baked into per_residue.
METRICS_JSON = ("../../data/analysis/Fig4/per_protein_analysis/md_1us_combined_48_justs2_justdihedral_panav10seedens.json")
REC_PATH = ("../../data/analysis/Fig4/sasa_results/per_residue_records.json")
# pre-computed for these same 48 trajectories (packing is a purely geometric feature of the
# structure, independent of the CS-prediction convention, so the existing cache is reused as-is)
PACK_CACHE = ("../../data/analysis/Fig4/per_protein_analysis/packN8_cache_combined_48.json")
DATA_PKL = f"{DATA}/ss_error_attribution_data_panav10seedens.pkl"
DATA_JSON = f"{DATA}/ss_error_attribution_data_panav10seedens.json"

DROP = 100
PACK_CUTOFF_NM = 0.8
PACK_STRIDE = 1

os.makedirs(FIG, exist_ok=True)
REC = json.load(open(REC_PATH))

In [ ]:
def frame_paths(entry):
    """Return the trajectory PDB frames for an entry, sorted numerically.

    Handles both naming conventions in this folder: frame_0001.pdb and frame0.pdb.
    """
    paths = glob.glob(f"{RES}/{entry}/frame*.pdb")

    def frame_number(path):
        digits = re.findall(r"(\d+)", os.path.basename(path))
        return int(digits[-1]) if digits else -1

    return sorted(paths, key=frame_number)

## 1. Dynamics and |error| from `md_1us_combined_48_justs2_justdihedral_panav10seedens.json`

In [ ]:
metrics = json.load(open(METRICS_JSON))["proteins"]

usable_entries = []
for entry in metrics:
    if len(frame_paths(entry)) == 0:
        continue
    usable_entries.append(entry)

print(f"proteins in JSON: {len(metrics)}")
print(f"usable (in JSON, has trajectory, not excluded): {len(usable_entries)}")


def build_reused_rows(metrics, entries):
    rows = []
    for entry in entries:
        top = metrics[entry]["per_residue"]
        for i in range(len(top["resnum"])):
            rows.append({
                "entry": entry,
                "resnum": top["resnum"][i],
                "aa": top["aa"][i],
                "ss": top["ss_class"][i],
                "exposure": top["exposure"][i],
                "acerr": top["acerr"][i],
                "s2_ired": top["s2_ired"][i],
            })
    return rows

data = pd.DataFrame(build_reused_rows(metrics, usable_entries))

print(f"\n{len(data)} residues total")
print("by secondary structure:")
print(data["ss"].value_counts(dropna=False).to_string())
print("\ncoverage (non-null):")
for column in ["acerr", "s2_ired"]:
    print(f"  {column}: {data[column].notna().sum()}")

## 2. Complexity `pack_N8`: heavy atoms within 8 Å of the amide N

In [ ]:
def compute_pack_N8(entry):
    """Mean number of heavy atoms within 8 A of each residue's amide N, over the trajectory."""
    trajectory = mdtraj.load(frame_paths(entry))[DROP:]
    trajectory = trajectory[::PACK_STRIDE]
    topology = trajectory.top
    heavy_atoms = np.array([atom.index for atom in topology.atoms if atom.element.symbol != "H"])
    records = REC[entry]
    n_residues = min(topology.n_residues, len(records))

    pack_by_resnum = {}
    for residue_index in range(n_residues):
        residue = topology.residue(residue_index)
        amide_nitrogen = [atom.index for atom in residue.atoms if atom.name == "N"]
        if not amide_nitrogen:
            continue
        neighbors = mdtraj.compute_neighbors(
            trajectory,
            PACK_CUTOFF_NM,
            np.array([amide_nitrogen[0]]),
            haystack_indices=heavy_atoms,
        )
        mean_count = np.mean([len(frame_neighbors) for frame_neighbors in neighbors]) + 1.0
        pack_by_resnum[str(records[residue_index]["resnum"])] = float(mean_count)
    return pack_by_resnum


if os.path.exists(PACK_CACHE):
    pack_data = json.load(open(PACK_CACHE))
    print(f"loaded pack_N8 cache ({len(pack_data)} proteins)")
else:
    pack_data = {}
    start_time = time.time()
    for entry in usable_entries:
        pack_data[entry] = compute_pack_N8(entry)
        print(f"  {entry} pack_N8 done [{time.time() - start_time:.0f}s]", flush=True)
    json.dump(pack_data, open(PACK_CACHE, "w"))
    print("wrote", PACK_CACHE)

## 3. Merge complexity into the table and save to pkl + json

In [ ]:
def lookup_pack_N8(row):
    per_protein = pack_data.get(row["entry"], {})
    return per_protein.get(str(row["resnum"]))


data["pack_N8"] = data.apply(lookup_pack_N8, axis=1)
# data.to_pickle(DATA_PKL)
data.to_json(DATA_JSON, orient="records", indent=1)

# print(f"saved {DATA_PKL}")
print(f"saved {DATA_JSON}")

In [ ]:
data["cls"] = data["exposure"] + "_" + data["ss"]

print(f"{len(data)} residues from {data.entry.nunique()} proteins, "
      f"{data.pack_N8.notna().sum()} with pack_N8")
print("\npack_N8 by exposure (mean):")
print(data.groupby("exposure")["pack_N8"].mean().round(1).to_string())

## 4. Complexity × dynamics matrix of the error

In [ ]:
N_BINS = 6

subset = data.dropna(subset=["s2_ired", "acerr"]).copy()

subset["s2_ired_bin"] = pd.qcut(
    subset["s2_ired"],
    q=N_BINS,
    labels=[f"Q{i+1}" for i in range(N_BINS)]
)

summary = (
    subset
    .groupby("s2_ired_bin", observed=True)["acerr"]
    .agg(["median", "count"])
)

print(summary)

In [ ]:
N_BINS = 6

subset = data.dropna(subset=["pack_N8", "acerr"]).copy()

subset["pack_N8_bin"] = pd.qcut(
    subset["pack_N8"],
    q=N_BINS,
    labels=[f"Q{i+1}" for i in range(N_BINS)]
)

summary = (
    subset
    .groupby("pack_N8_bin", observed=True)["acerr"]
    .agg(["median", "count"])
)

print(summary)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def set_style(base=12, family='Arial'):
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        'font.family': family, 'font.size': base,
        'axes.titlesize': base + 2, 'axes.labelsize': base,
        'xtick.labelsize': base - 2, 'ytick.labelsize': base - 2,
        'legend.fontsize': base - 2, 'svg.fonttype': 'none',
    })

set_style(base=16, family='Arial')
N_BINS = 4

# ---- shared quantile edges (computed ONCE so both panels use identical bins) ----
base = data.dropna(subset=["pack_N8", "s2_ired"]).copy()
_, cx_edges = pd.qcut(base["pack_N8"], N_BINS, labels=False, retbins=True)
_, dy_edges = pd.qcut(base["s2_ired"], N_BINS, labels=False, retbins=True)


def bin_labels(edges, fmt):
    return [f"{edges[i]:{fmt}}-{edges[i + 1]:{fmt}}" for i in range(len(edges) - 1)]


cx_labels = bin_labels(cx_edges, ".0f")     # pack_N8: integer-ish
dy_labels = bin_labels(dy_edges, ".2f")     # 1 - s2_ired: two decimals

print("complexity  pack_N8  quartile ranges:")
for i, label in enumerate(cx_labels):
    print(f"  Q{i + 1}: {label}")
print("dynamics  1 - s2_ired  quartile ranges:")
for i, label in enumerate(dy_labels):
    print(f"  Q{i + 1}: {label}")


def build_matrix(df, value_column, cx_edges, dy_edges, n_bins=N_BINS):
    subset = df.dropna(subset=["pack_N8", "s2_ired", value_column]).copy()
    # assign to the SHARED edges (include_lowest so the minimum value lands in Q1)
    subset["complexity_bin"] = pd.cut(subset["pack_N8"], cx_edges, labels=False, include_lowest=True)
    subset["dynamics_bin"] = pd.cut(subset["s2_ired"], dy_edges, labels=False, include_lowest=True)

    matrix = np.full((n_bins, n_bins), np.nan)
    counts = np.zeros((n_bins, n_bins), dtype=int)
    for complexity_bin in range(n_bins):
        for dynamics_bin in range(n_bins):
            cell = subset[
                (subset["complexity_bin"] == complexity_bin)
                & (subset["dynamics_bin"] == dynamics_bin)
            ]
            if len(cell) >= 5:
                matrix[dynamics_bin, complexity_bin] = cell[value_column].median()
                counts[dynamics_bin, complexity_bin] = len(cell)
    return matrix, counts

HM_SIZE = 3.9                                   # heatmap edge length in inches, identical for every model
figure = plt.figure(figsize=(8, 8))             # generous canvas; savefig(bbox_inches="tight") crops it
ax = figure.add_axes([0.25, 0.25, HM_SIZE / 8, HM_SIZE / 8])   # axes box = exactly HM_SIZE x HM_SIZE in

matrix, counts = build_matrix(data, "acerr", cx_edges, dy_edges)
image = ax.imshow(matrix, origin="lower", cmap="viridis_r", aspect="auto", vmin=0.65, vmax=1.5)
for dynamics_bin in range(N_BINS):
    for complexity_bin in range(N_BINS):
        value = matrix[dynamics_bin, complexity_bin]
        if np.isfinite(value):
            ax.text(
                complexity_bin,
                dynamics_bin,
                f"{value:.2f}\nn={counts[dynamics_bin, complexity_bin]}",
                ha="center",
                va="center",
                color="white" if value > 1.05 else "black",
            )
# ax.set_xticks(range(N_BINS))
# ax.set_xticklabels(cx_labels, ha="center")
# ax.set_yticks(range(N_BINS))
# ax.set_yticklabels(dy_labels)
# ax.set_xlabel("complexity  pack_N8  (low -> high)")
# ax.set_ylabel("dynamics S2  (high -> low)")
# ax.set_title("SLAE-CS")
# cax = figure.add_axes([0.25 + (HM_SIZE + 0.2) / 8, 0.25 + 0.1 * HM_SIZE / 8,
#                        0.2 / 8, 0.8 * HM_SIZE / 8])
ax.set_xticks(range(N_BINS))
ax.set_xticklabels(cx_labels, ha="center")
ax.set_yticks(range(N_BINS))
ax.set_yticklabels(dy_labels)
ax.set_xlabel("complexity  pack_N8  (low -> high)")
ax.set_ylabel("dynamics S2  (high -> low)")
ax.set_title("SLAE-CS")
# ax.text(0.5, -0.15, "cell: median |error| (ppm)\nn = count of residues", transform=ax.transAxes,
#         ha="center", va="top", fontsize=13, color="0.35", font="Arial")
# cb = figure.colorbar(image, cax=cax)
# cb.ax.set_ylabel("median |error| (ppm)", rotation=270, va="bottom")

figure.canvas.draw()                                            # lay out everything drawn so far
renderer = figure.canvas.get_renderer()
fw, fh = figure.get_size_inches()
x_left = ax.yaxis.get_tightbbox(renderer).x0 / (fw * figure.dpi)   # left edge of ylabel + yticklabels
y_top = ax.get_position().y1                                    # top edge of the heatmap box
SHIFT_IN = 0.08                                                 # gap between heatmap top and note (inches)
# figure.text(x_left, y_top + SHIFT_IN / fh, "cell: median |error| (ppm)\nn = count of residues",
#             ha="left", va="bottom", multialignment="left", fontsize=13, color="0.35", font="Arial")

# figure.suptitle("Complexity x dynamics matrix  (each cell = median over residues)", y=1.02)
plt.tight_layout()
figure.savefig(f"{FIG}/ss_error_attribution_matrix_justerror_panav10seedens.png", dpi=300, bbox_inches="tight")
figure.savefig(f"{FIG}/ss_error_attribution_matrix_justerror_panav10seedens.svg", bbox_inches="tight")
plt.show()
print("saved ss_error_attribution_matrix_panav10seedens.png/.svg")